# Kế hoạch ETL - Staging: OULAD Dataset
Mục tiêu của Notebook này là xử lý dữ liệu thô (raw_data), thực hiện các bước kiểm kê, gộp bảng, loại bỏ dữ liệu dư thừa và chuẩn hóa kiểu dữ liệu. Sau đó, dữ liệu sẽ được lưu vào một thư mục staging để sẵn sàng cho quá trình Khám phá Dữ liệu (EDA).


## 1. Kiểm kê dữ liệu
**Ý nghĩa:**
Bước này giúp chúng ta có cái nhìn tổng quan về các file dữ liệu có trong `raw_data`. Bằng cách liệt kê và kiểm tra số dòng, số cột, tên cột, chúng ta sẽ biết được quy mô của bộ dữ liệu, xác nhận tính toàn vẹn (đủ 7 bảng chính hay chưa) và phát hiện sớm các bất thường về mặt cấu trúc trước khi load toàn bộ vào bộ nhớ.

**Công việc cụ thể:**
- Liệt kê toàn bộ file CSV trong `raw_data`.
- Xác nhận đủ 7 bảng chính của OULAD.
- Ghi nhận các file bị chia nhỏ như `studentVle_0.csv` đến `studentVle_7.csv`.
- Kiểm tra số dòng, số cột và tên cột của từng bảng.
- Tóm tắt vai trò của từng bảng trong dữ liệu.


## 2. Load dữ liệu thô
**Ý nghĩa:**
Đảm bảo việc đọc dữ liệu được thực hiện một cách nhất quán. Việc tuân thủ nguyên tắc không chỉnh sửa trực tiếp file CSV gốc giúp bảo toàn nguồn dữ liệu, cho phép chạy lại quy trình ETL bất cứ lúc nào.

**Công việc cụ thể:**
- Đọc từng file CSV bằng cấu hình thống nhất.
- Giữ nguyên dữ liệu gốc trong `raw_data`.
- Không chỉnh sửa trực tiếp file CSV gốc.
- Xác định nơi lưu dữ liệu staging hoặc dữ liệu trung gian.
- Ghi nhận lỗi đọc file nếu có.


## 3. Gộp bảng bị chia nhỏ
**Ý nghĩa:**
Do kích thước lớn, bảng `studentVle` bị chia thành 8 file nhỏ. Gộp chúng lại thành một DataFrame duy nhất giúp quá trình phân tích sau này không bị phân mảnh. Việc kiểm tra trước/sau và kiểm tra duplicate đảm bảo quá trình gộp diễn ra chính xác.

**Công việc cụ thể:**
- Gộp `studentVle_0.csv` đến `studentVle_7.csv` thành một bảng logic `studentVle`.
- Kiểm tra các file `studentVle` có cùng cấu trúc cột.
- Kiểm tra tổng số dòng trước và sau khi gộp.
- Kiểm tra duplicate sau khi gộp.
- Lưu hoặc sinh bảng `studentVle` đã gộp cho bước EDA.


## 4. Loại bỏ cột kỹ thuật dư thừa
**Ý nghĩa:**
Một số file CSV khi xuất ra có thể đi kèm cột index mặc định của hệ thống cũ (như `Unnamed: 0`). Các cột này không chứa thông tin nghiệp vụ và gây nhiễu khi phân tích. Việc loại bỏ chúng giúp làm sạch schema của bảng.

**Công việc cụ thể:**
- Kiểm tra các cột index không tên như `Unnamed: 0`.
- Loại bỏ cột index dư khỏi bảng dùng để phân tích.
- Xác nhận cột bị loại bỏ không mang ý nghĩa nghiệp vụ.
- Ghi lại quy tắc loại bỏ cột kỹ thuật.


## 5. Chuẩn hóa kiểu dữ liệu
**Ý nghĩa:**
Đảm bảo mỗi cột có kiểu dữ liệu phản ánh đúng bản chất của nó. Ví dụ: cột categorical nên được chuyển sang kiểu phân loại, các cột số ở đúng định dạng để tính toán. Điều này cũng giúp tiết kiệm bộ nhớ khi xử lý.

**Công việc cụ thể:**
- Chuyển `id_student`, `id_site`, `id_assessment` về kiểu phù hợp.
- Xem `code_module`, `code_presentation`, `assessment_type`, `final_result` là biến phân loại.
- Xử lý các cột ngày tương đối như `date`, `date_submitted`, `date_registration`, `date_unregistration`.
- Chuyển `score`, `weight`, `sum_click` về kiểu số.
- Kiểm tra lại kiểu dữ liệu sau khi gộp dữ liệu.


## 6. Đánh giá chất lượng dữ liệu sau Staging
Đoạn code dưới đây đọc lại dữ liệu từ `staging_data`, kiểm tra kiểu dữ liệu (data types), đếm số lượng giá trị thiếu (missing values) và xem khoảng giá trị (value range) của từng cột. Kết quả sẽ được ghi nối tiếp vào file `etl_report.md`.

In [ ]:
import os
import pandas as pd

staging_dir = r"d:\Học Tập\2026.1\Kho dữ liệu và kinh doanh thông minh\oulad-data-warehouse\ETL\staging_data"
report_path = r"d:\Học Tập\2026.1\Kho dữ liệu và kinh doanh thông minh\oulad-data-warehouse\ETL\etl_report.md"

tables = ['courses', 'assessments', 'vle', 'studentInfo', 'studentRegistration', 'studentAssessment', 'studentVle']
report_lines = ["\n## 5. Đánh giá chất lượng dữ liệu (Data Type & Value Range)"]

for t in tables:
    file_path = os.path.join(staging_dir, f"{t}.csv")
    if os.path.exists(file_path):
        # Đọc dữ liệu (pandas tự suy luận kiểu dựa trên CSV)
        df = pd.read_csv(file_path, low_memory=False)
        report_lines.append(f"\n### Bảng: {t}")
        report_lines.append("| Cột | Kiểu dữ liệu | Số lượng Non-Null | Missing (%) | Min / Max / Unique |")
        report_lines.append("|---|---|---|---|---|")
        
        for col in df.columns:
            dtype = str(df[col].dtype)
            non_null = df[col].notnull().sum()
            missing_pct = round(df[col].isnull().sum() / len(df) * 100, 2)
            
            if pd.api.types.is_numeric_dtype(df[col]):
                min_val = round(df[col].min(), 2) if pd.notnull(df[col].min()) else "NaN"
                max_val = round(df[col].max(), 2) if pd.notnull(df[col].max()) else "NaN"
                range_str = f"{min_val} -> {max_val}"
            else:
                num_unique = df[col].nunique()
                range_str = f"{num_unique} giá trị (unique)"
                
            report_lines.append(f"| {col} | {dtype} | {non_null} | {missing_pct}% | {range_str} |")

with open(report_path, "a", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print("Đã bổ sung báo cáo chất lượng dữ liệu vào etl_report.md")
